<a href="https://colab.research.google.com/github/ymuto0302/ML_Study_Session/blob/main/sklearn_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## データの準備

### scikit-learn のデータ形式

In [1]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data      # (150, 4) — NumPy 配列
y = iris.target    # (150,)

print(f"特徴量行列: {X.shape}")
print(f"特徴量名: {iris.feature_names}")
print(f"目的変数: {y.shape}")
print(f"クラス名: {iris.target_names}")

特徴量行列: (150, 4)
特徴量名: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
目的変数: (150,)
クラス名: ['setosa' 'versicolor' 'virginica']


### 合成(人工)データの生成

In [2]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# 分類タスク用の合成データ
X_cls, y_cls = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_classes=3, random_state=42
)

# 回帰タスク用の合成データ
X_reg, y_reg = make_regression(
    n_samples=1000, n_features=10, noise=10, random_state=42
)

# クラスタリング用の合成データ
X_blobs, y_blobs = make_blobs(
    n_samples=300, centers=4, cluster_std=1.0, random_state=42
)

### Pandas DataFrame からの変換

In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 一旦，Iris データセットを Pandas DataFrame に格納
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y

# Pandas DataFrame から NumPy 配列に変換 するには .values を用いる
X_from_df = df.drop('target', axis=1).values
y_from_df = df['target'].values

model = RandomForestClassifier(n_estimators=100)
model.fit(X_from_df, y_from_df)

# scikit-learn は pandas DataFrame も直接受け取れる（バージョン 1.0 以降推奨）
# model.fit(df.drop('target', axis=1), df['target']) も可

RandomForestClassifier()

### データの分割

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,       # テストデータの割合
    random_state=42,      # 再現性のためのシード
    stratify=y            # 層化抽出（分類の場合は常に推奨）
)

print(f"訓練: {X_train.shape}, テスト: {X_test.shape}")

訓練: (105, 4), テスト: (45, 4)


---
## 前処理

### 特徴量のスケーリング

In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 標準化（平均 0、標準偏差 1）
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # 訓練データで fit + transform
X_test_sc = scaler.transform(X_test)        # テストデータは transform のみ！

# Min-Max 正規化（[0, 1] の範囲）
minmax = MinMaxScaler()
X_train_mm = minmax.fit_transform(X_train)
X_test_mm = minmax.transform(X_test)


### カテゴリカル変数のエンコーディング

In [8]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# ラベルエンコーディング（目的変数向け）
le = LabelEncoder()
y_encoded = le.fit_transform(['cat', 'dog', 'cat', 'bird'])
print(f"エンコード: {y_encoded}")            # [1, 2, 1, 0]
print(f"逆変換: {le.inverse_transform(y_encoded)}")  # ['cat', 'dog', 'cat', 'bird']

# One-Hot エンコーディング（特徴量向け）
ohe = OneHotEncoder(sparse_output=False)
X_cat = np.array([['red'], ['blue'], ['red'], ['green']])
X_encoded = ohe.fit_transform(X_cat)
print(f"\nOne-Hot:\n{X_encoded}")

エンコード: [1 2 1 0]
逆変換: ['cat' 'dog' 'cat' 'bird']

One-Hot:
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


### 欠損値の補完

In [9]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy='mean')  # 'median', 'most_frequent', 'constant' も可
X_imputed = imp.fit_transform(X_train)


In [10]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_train[:5, :2])  # 2 特徴量を 2 次多項式に展開
print(f"元の特徴量数: 2 → 多項式展開後: {X_poly.shape[1]}")
# x1, x2, x1^2, x1*x2, x2^2 → 5 特徴量


元の特徴量数: 2 → 多項式展開後: 5


---
## 主要な学習アルゴリズム

### 分類タスク

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# すべて同じ API
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5),
    'Random Forest':       RandomForestClassifier(n_estimators=100),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100),
    'SVM':                 SVC(kernel='rbf'),
    'k-NN':                KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':         GaussianNB(),
    'MLP':                 MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300),
}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    score = model.score(X_test_sc, y_test)
    print(f"{name:<25s}: {score:.4f}")

Logistic Regression      : 0.9111
Decision Tree            : 0.9778
Random Forest            : 0.9333
Gradient Boosting        : 0.9333
SVM                      : 0.9333
k-NN                     : 0.9111
Naive Bayes              : 0.9111
MLP                      : 0.9111


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


### 回帰タスク

In [12]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# 回帰も同じ API
# model.fit(X_train, y_train)
# model.predict(X_test)
# model.score(X_test, y_test)  → R² を返す


### クラスタリング

In [13]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# クラスタリングは y（ラベル）なしで学習
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)  # fit と predict を同時に


### 次元削減

In [14]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)


---
## パイプライン

In [15]:
from sklearn.pipeline import Pipeline

# 前処理 → モデルのパイプライン
pipe = Pipeline([
    ('scaler', StandardScaler()),           # ステップ 1: 標準化
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))  # ステップ 2: 分類
])

# fit は内部で scaler.fit_transform → clf.fit を順に実行
pipe.fit(X_train, y_train)

# predict は内部で scaler.transform → clf.predict を順に実行
y_pred = pipe.predict(X_test)

print(f"パイプライン精度: {pipe.score(X_test, y_test):.4f}")


パイプライン精度: 0.8889


---
## モデル選択とハイパーパラメータチューニング

### 交差検証

In [16]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
print(f"5-Fold CV: {scores.mean():.4f} ± {scores.std():.4f}")


5-Fold CV: 0.9467 ± 0.0267


### グリッドサーチ

In [17]:
from sklearn.model_selection import GridSearchCV

# パイプライン内のパラメータは "ステップ名__パラメータ名" で指定
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [3, 5, 10, None],
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,           # 全 CPU コアで並列実行
    return_train_score=True
)
grid.fit(X_train, y_train)

print(f"最良パラメータ: {grid.best_params_}")
print(f"最良 CV スコア: {grid.best_score_:.4f}")
print(f"テストスコア:   {grid.score(X_test, y_test):.4f}")


最良パラメータ: {'clf__max_depth': 3, 'clf__n_estimators': 50}
最良 CV スコア: 0.9524
テストスコア:   0.8889


### ランダムサーチ

In [18]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_distributions = {
    'clf__n_estimators': randint(50, 300),
    'clf__max_depth': [3, 5, 10, 20, None],
    'clf__min_samples_leaf': randint(1, 10),
}

random_search = RandomizedSearchCV(
    pipe,
    param_distributions,
    n_iter=30,           # 30 回のランダムサンプリング
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)

print(f"最良パラメータ: {random_search.best_params_}")
print(f"最良 CV スコア: {random_search.best_score_:.4f}")


最良パラメータ: {'clf__max_depth': 20, 'clf__min_samples_leaf': 8, 'clf__n_estimators': 238}
最良 CV スコア: 0.9619


---
## モデルの評価

### 分類タスクの評価

In [19]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

y_pred = grid.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1:        {f1_score(y_test, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Accuracy:  0.8889
Precision: 0.8981
Recall:    0.8889
F1:        0.8878

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.78      0.93      0.85        15
   virginica       0.92      0.73      0.81        15

    accuracy                           0.89        45
   macro avg       0.90      0.89      0.89        45
weighted avg       0.90      0.89      0.89        45



### 回帰タスクの評価

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# y_pred_reg = model.predict(X_test)
# print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_reg)):.4f}")
# print(f"MAE:  {mean_absolute_error(y_test, y_pred_reg):.4f}")
# print(f"R²:   {r2_score(y_test, y_pred_reg):.4f}")


---
## モデルの保存と読み込み

In [21]:
import joblib

# モデルの保存
joblib.dump(grid.best_estimator_, 'best_model.pkl')

# モデルの読み込み
loaded_model = joblib.load('best_model.pkl')
print(f"読み込んだモデル: {loaded_model.score(X_test, y_test):.4f}")


読み込んだモデル: 0.8889


---
## 典型的なワークフローの完全なコード

In [22]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# 1. データの準備
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. パイプラインの構築
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

# 3. ハイパーパラメータチューニング
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [5, 10, None],
    'clf__min_samples_leaf': [1, 2, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

# 4. 最終評価
print(f"最良パラメータ: {grid.best_params_}")
print(f"最良 CV スコア (F1): {grid.best_score_:.4f}")
print()
y_pred = grid.predict(X_test)
print(classification_report(y_test, y_pred, target_names=data.target_names))

# 5. モデルの保存
joblib.dump(grid.best_estimator_, 'breast_cancer_model.pkl')


最良パラメータ: {'clf__max_depth': 5, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 100}
最良 CV スコア (F1): 0.9702

              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



['breast_cancer_model.pkl']